> Cập nhật 07/11: notebook mẫu chưa lưu output. Không dùng các nhận xét bên dưới làm kết quả thực nghiệm. Xem historical_eda_verified.ipynb cho EDA đã chạy trên snapshot của model.

# 📊 01. Phân Tích Dữ Liệu Khám Phá (EDA) & Kiểm Toán Chất Lượng Dữ Liệu
**Dự án**: Streaming MLOps E-Commerce — Dự Báo Nhu Cầu Hàng Hóa
**Giai đoạn**: Tuần 4 — Tiền xử lý dữ liệu & Khám phá phân bố

---
### Mục tiêu nghiên cứu:
1. **Kiểm toán chất lượng dữ liệu (Data Quality Audit)**: Lập bảng đối chiếu toàn diện số liệu trước và sau khi làm sạch theo đúng quy chuẩn Cẩm nang ĐATN.
2. **Phân tích kênh bán hàng (Channel Dynamics)**: So sánh hành vi mua hàng giữa Shopee (55%) và TikTok Shop (45%).
3. **Phân tích chu kỳ thời gian & Mùa vụ**: Khám phá đỉnh mua hàng theo giờ (Flash Sale), ngày trong tuần, và hiệu ứng bùng nổ đơn ngày đôi (Mega-Sale: 8/8, 9/9, 10/10).
4. **Phân tích độ nhạy cảm giá & Chiết khấu**: Xác định tương quan giữa giá, voucher, phí ship và sản lượng bán (`quantity`).


In [ ]:
# 1. Thiết lập môi trường và cấu hình đồ họa
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình font và palette màu hiện đại
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.figsize'] = (12, 6)
palette = sns.color_palette('tab10')

# Thêm đường dẫn gốc dự án để import các module
sys.path.insert(0, os.path.abspath('..'))
from warehouse.etl.transform import unify_schema, clean_data, generate_quality_report
from warehouse.etl.data_validation import DataValidator


### 2. Tải dữ liệu đa kênh (Data Loading)
Dữ liệu được nạp từ tập lịch sử đa kênh (hoặc sinh trực tiếp qua `ECommerceSimulator` nếu chưa có file local).


In [ ]:
# Tải dữ liệu hoặc sinh dữ liệu mẫu nếu chưa có file
data_path = os.path.join('..', 'data', 'historical_orders_raw.parquet')

if os.path.exists(data_path):
    raw_df = pd.read_parquet(data_path)
    print(f'✓ Đã tải {len(raw_df):,} dòng dữ liệu thô từ file Parquet.')
else:
    print('! Không tìm thấy file Parquet có sẵn, tiến hành sinh 10,000 đơn mẫu đa kênh...')
    from scripts.simulate_historical_data import generate_historical_orders
    raw_df = generate_historical_orders(days=45, base_orders_per_day=150)

print('Cột dữ liệu thô:', raw_df.columns.tolist()[:10], '...')
raw_df.head(2)


### 3. Kiểm toán chất lượng dữ liệu: Bảng đối chiếu Trước vs Sau Làm Sạch
Theo yêu cầu của Cẩm nang ĐATN, đối chiếu minh bạch các chỉ số trước và sau khi thực hiện pipeline ETL:
- Số lượng bản ghi và tỷ lệ bản ghi trùng lặp.
- Tỷ lệ giá trị khuyết (Missing Values) theo từng cột quan trọng.
- Thống kê mô tả (Mean, Median, Std, Min, Max) của các biến số chính.


In [ ]:
# Thực hiện Unify schema và Clean data
unified_df = unify_schema(raw_df)
clean_df = clean_data(unified_df)

# 1. Đối chiếu số lượng bản ghi
audit_counts = pd.DataFrame({
    'Chỉ số': ['Tổng số bản ghi ban đầu', 'Bản ghi sau Unify', 'Bản ghi sau khử trùng lặp', 'Số bản ghi trùng lặp đã loại bỏ'],
    'Giá trị': [len(raw_df), len(unified_df), len(clean_df), len(unified_df) - len(clean_df)]
})
display(audit_counts)

# 2. Đối chiếu tỷ lệ giá trị khuyết
cols_to_check = ['order_id', 'sku', 'product_name', 'quantity', 'original_price', 'buyer_total_amount', 'shipping_carrier']
missing_before = unified_df[cols_to_check].isnull().mean() * 100
missing_after = clean_df[cols_to_check].isnull().mean() * 100
audit_missing = pd.DataFrame({
    'Cột': cols_to_check,
    '% Khuyết trước Clean': missing_before.values.round(2),
    '% Khuyết sau Clean': missing_after.values.round(2)
})
display(audit_missing)

# 3. Thống kê mô tả trước vs sau
metrics = ['quantity', 'original_price', 'buyer_total_amount']
stats_summary = pd.concat([
    clean_df[metrics].describe().T[['mean', 'std', 'min', '50%', 'max']].rename(columns={'50%': 'median'})
], axis=1).round(2)
print('Thống kê mô tả tập dữ liệu sạch:')
display(stats_summary)


**Cần kiểm chứng sau khi chạy:** đối chiếu khóa thiếu, số dòng bị loại và miền giá trị từ output thật. Chưa có output lưu trong notebook này để xác nhận 100% dữ liệu sạch.

### 4. Phân tích phân phối theo kênh bán & Trạng thái đơn hàng
So sánh đặc tính đơn hàng giữa hai nền tảng Shopee và TikTok Shop.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Tỷ lệ đơn theo sàn
platform_counts = clean_df['platform'].value_counts()
axes[0].pie(platform_counts, labels=['Shopee', 'TikTok Shop'], autopct='%1.1f%%',
            colors=['#EE4D2D', '#000000'], startangle=90, explode=(0.03, 0.03))
axes[0].set_title('Tỷ trọng đơn hàng theo sàn TMĐT')

# Phân bố trạng thái đơn hàng
sns.countplot(data=clean_df, y='order_status', hue='platform', ax=axes[1], palette=['#EE4D2D', '#00F2FE'])
axes[1].set_title('Phân bố trạng thái đơn hàng')
axes[1].set_xlabel('Số lượng đơn')
axes[1].set_ylabel('Trạng thái')
plt.tight_layout()
plt.show()

cancel_rate = clean_df['is_cancelled'].mean() * 100
print(f'Tỷ lệ hủy đơn bình quân trên toàn hệ thống: {cancel_rate:.2f}%')


**Giới hạn diễn giải:** tỷ lệ kênh của simulator là giả định cấu hình, không phải ước lượng thị phần Việt Nam. Tỷ lệ hủy phải được tính từ dữ liệu của lần chạy, không ghi trước 12%.

### 5. Phân tích chuỗi thời gian & Tính mùa vụ (Seasonality)
Kiểm chứng các quy luật mua sắm đặc trưng: giờ vàng Flash Sale, ngày cuối tuần và ngày đôi Mega-sale.


In [ ]:
clean_df['dt'] = pd.to_datetime(clean_df['create_time'])
clean_df['hour'] = clean_df['dt'].dt.hour
clean_df['day_of_week'] = clean_df['dt'].dt.day_name()
clean_df['day_num'] = clean_df['dt'].dt.dayofweek

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Phân bố đơn theo giờ trong ngày
hourly_orders = clean_df.groupby('hour')['quantity'].sum()
sns.lineplot(x=hourly_orders.index, y=hourly_orders.values, marker='o', ax=axes[0], color='#2E86AB', linewidth=2.5)
axes[0].axvspan(11.5, 13.5, color='#F6D55C', alpha=0.3, label='Flash Sale 12h Trưa')
axes[0].axvspan(19.5, 21.5, color='#ED553B', alpha=0.3, label='Flash Sale 20-21h Tối')
axes[0].set_title('Tổng sản lượng đơn hàng theo giờ trong ngày')
axes[0].set_xlabel('Giờ (0 - 23h)')
axes[0].set_ylabel('Sản lượng (Quantity)')
axes[0].legend()

# Phân bố đơn theo ngày trong tuần
order_days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_orders = clean_df.groupby('day_of_week')['quantity'].sum().reindex(order_days)
sns.barplot(x=dow_orders.index, y=dow_orders.values, ax=axes[1], palette='Blues_d')
axes[1].set_title('Tổng sản lượng đơn hàng theo ngày trong tuần')
axes[1].set_xlabel('Thứ')
axes[1].set_ylabel('Sản lượng (Quantity)')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()


**Cần kiểm chứng sau khi chạy:** chỉ kết luận đỉnh theo giờ hoặc tăng cuối tuần khi có biểu đồ và số đo. Các hệ số mô phỏng không chứng minh hiệu ứng thị trường thực tế.

### 6. Phân tích tương quan & Độ nhạy cảm giá
Ma trận tương quan giữa số lượng mua, giá gốc, chiết khấu và phí vận chuyển.


In [ ]:
corr_cols = ['quantity', 'original_price', 'buyer_total_amount', 'seller_discount', 'platform_discount', 'shipping_fee']
existing_corr_cols = [c for c in corr_cols if c in clean_df.columns]

corr_matrix = clean_df[existing_corr_cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, vmin=-1, vmax=1)
plt.title('Ma trận tương quan (Correlation Matrix) giữa các biến số tài chính')
plt.tight_layout()
plt.show()


### Kết luận và giới hạn
Notebook này là mẫu phân tích simulator và chưa lưu kết quả chạy. Không dùng nhận xét dự kiến thay cho EDA thực nghiệm. Xem historical_eda_verified.ipynb để phân tích snapshot lịch sử cùng hạn chế zero-fill, thiếu category và coverage.